In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("sp500_headlines_with_prices sep ~.csv")
df["Date"] = pd.to_datetime(df["Date"])
df.head()


[*********************100%***********************]  1 of 1 completed


,Date,combined_headline,headline_count,close_price,open_price,high_price,low_price,volume
0,2008-01-02,"JPMorgan Predicts 2008 Will Be ""Nothing But Ne...",3,1447.16,1467.97,1471.77,1442.07,1918138889
1,2008-01-03,"U.S. Stocks Higher After Economic Data, Monsan...",1,1447.16,1447.55,1456.80,1443.73,1905277778
2,2008-01-07,U.S. Stocks Climb As Hopes Increase For More F...,1,1416.18,1414.07,1423.87,1403.45,2345144444
3,2008-01-09,How Investing in Intangibles -- Like Employee ...,2,1409.13,1390.25,1409.19,1378.70,2972794444
4,2008-01-10,U.S. Stocks Zigzag Higher As Bernanke Speech S...,1,1420.33,1406.78,1429.09,1395.31,2872494444


In [51]:
sorted_df = df.sort_values(by="Date").reset_index(drop=True)
sorted_df["next_day_price"] = sorted_df["open_price"].shift(-1)
sorted_df["target"] = (sorted_df["close_price"] < sorted_df["next_day_price"]).astype(int)
sorted_df.head()
sorted_df = sorted_df.dropna(subset=["next_day_price"]).reset_index(drop=True)

In [52]:
from sklearn.model_selection import train_test_split
X = sorted_df['combined_headline']
Y = sorted_df['target']
#Don't shuffle or else the model will get access to future data.
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size=.2, shuffle=False)

print("If markets are always down accuracy:", (Y_test==0).mean())
print("If markets are always up accuracy:", (Y_test ==1).mean())

If markets are always down accuracy: 0.46438746438746437
If markets are always up accuracy: 0.5356125356125356


In [75]:
#https://scikit-learn.org/1.4/tutorial/text_analytics/working_with_text_data.html
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

vectorizer = CountVectorizer(ngram_range=(1,2), stop_words=('english'))
X_train_counts = vectorizer.fit_transform(X_train)
transformer = TfidfTransformer(use_idf=True)
X_train_tfidf = transformer.fit_transform(X_train_counts)

from sklearn.svm import LinearSVC
clf = LinearSVC().fit(X_train_tfidf, Y_train)

In [76]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
X_test_counts = vectorizer.transform(X_test)
X_test_tfidf = transformer.transform(X_test_counts)
predictions = clf.predict(X_test_tfidf)

print("Accuracy: ", accuracy_score(Y_test, predictions), "\n")
print("Classification Report: ", classification_report(Y_test, predictions), "\n")
print("Confusion Matrix: ", confusion_matrix(Y_test, predictions))

Accuracy:  0.5370370370370371 

Classification Report:                precision    recall  f1-score   support

           0       0.50      0.18      0.26       326
           1       0.54      0.85      0.66       376

    accuracy                           0.54       702
   macro avg       0.52      0.51      0.46       702
weighted avg       0.53      0.54      0.48       702
 

Confusion Matrix:  [[ 58 268]
 [ 57 319]]


Accuracy:  0.5185185185185185 

Classification Report:                precision    recall  f1-score   support

           0       0.47      0.27      0.34       326
           1       0.54      0.73      0.62       376

    accuracy                           0.52       702
   macro avg       0.50      0.50      0.48       702
weighted avg       0.51      0.52      0.49       702
 

Confusion Matrix:  [[ 89 237]
 [101 275]]
 ## Bigrams + stop words
 Accuracy:  0.5370370370370371 

Classification Report:                precision    recall  f1-score   support

           0       0.50      0.18      0.26       326
           1       0.54      0.85      0.66       376

    accuracy                           0.54       702
   macro avg       0.52      0.51      0.46       702
weighted avg       0.53      0.54      0.48       702
 

Confusion Matrix:  [[ 58 268]